# Data Cleaning and Preprocessing for Twitter Sentiment Analysis

## Objective

The objective of this notebook is to preprocess demonetization-related tweets for sentiment analysis.

Tweet preprocessing is important because social media text contains:
- hashtags
- mentions
- URLs
- punctuation
- emojis
- noisy text

This notebook prepares clean textual data for sentiment analysis using:
- VADER
- RoBERTa

In [1]:
# Data Handling Libraries
import pandas as pd
import numpy as np

# Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing Libraries
import re
import string
import nltk

# NLTK Modules
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Ignore Warnings
import warnings
warnings.filterwarnings('ignore')

# Plot Style
plt.style.use('ggplot')

print("Libraries Imported Successfully 🚀")

Libraries Imported Successfully 🚀


In [5]:
# Load Dataset
df = pd.read_csv("C:/Users/rahul/Desktop/demonetization-tweets.csv - Copy/demonetization-tweets.csv",encoding='latin1')

print("Dataset Loaded Successfully ✅")
print(df.shape)
df.head()

Dataset Loaded Successfully ✅
(14940, 16)


,Unnamed: 0,X,text,favorited,favoriteCount,replyToSN,created,truncated,replyToSID,id,replyToUID,statusSource,screenName,retweetCount,isRetweet,retweeted
0,1,1,RT @rssurjewala: Critical question: Was PayTM ...,False,0,NaN,2016-11-23 18:40:30,False,NaN,8.014957e+17,NaN,"<a href=""http://twitter.com/download/android"" ...",HASHTAGFARZIWAL,331,True,False
1,2,2,RT @Hemant_80: Did you vote on #Demonetization...,False,0,NaN,2016-11-23 18:40:29,False,NaN,8.014957e+17,NaN,"<a href=""http://twitter.com/download/android"" ...",PRAMODKAUSHIK9,66,True,False
2,3,3,"RT @roshankar: Former FinSec, RBI Dy Governor,...",False,0,NaN,2016-11-23 18:40:03,False,NaN,8.014955e+17,NaN,"<a href=""http://twitter.com/download/android"" ...",rahulja13034944,12,True,False
3,4,4,RT @ANI_news: Gurugram (Haryana): Post office ...,False,0,NaN,2016-11-23 18:39:59,False,NaN,8.014955e+17,NaN,"<a href=""http://twitter.com/download/android"" ...",deeptiyvd,338,True,False
4,5,5,RT @satishacharya: Reddy Wedding! @mail_today ...,False,0,NaN,2016-11-23 18:39:39,False,NaN,8.014954e+17,NaN,"<a href=""http://cpimharyana.com"" rel=""nofollow...",CPIMBadli,120,True,False


In [11]:
# Display Column Names
df.columns


Index(['Unnamed: 0', 'X', 'text', 'favorited', 'favoriteCount', 'replyToSN',
       'created', 'truncated', 'replyToSID', 'id', 'replyToUID',
       'statusSource', 'screenName', 'retweetCount', 'isRetweet', 'retweeted'],
      dtype='object')

In [10]:
# Dataset Information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14940 entries, 0 to 14939
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     14940 non-null  int64  
 1   X              14940 non-null  int64  
 2   text           14940 non-null  object 
 3   favorited      14940 non-null  bool   
 4   favoriteCount  14940 non-null  int64  
 5   replyToSN      1102 non-null   object 
 6   created        14940 non-null  object 
 7   truncated      14940 non-null  bool   
 8   replyToSID     886 non-null    float64
 9   id             14940 non-null  float64
 10  replyToUID     1102 non-null   float64
 11  statusSource   14940 non-null  object 
 12  screenName     14940 non-null  object 
 13  retweetCount   14940 non-null  int64  
 14  isRetweet      14940 non-null  bool   
 15  retweeted      14940 non-null  bool   
dtypes: bool(4), float64(3), int64(4), object(5)
memory usage: 1.4+ MB


In [12]:
# Check Missing Values
df.isnull().sum()

Unnamed: 0           0
X                    0
text                 0
favorited            0
favoriteCount        0
replyToSN        13838
created              0
truncated            0
replyToSID       14054
id                   0
replyToUID       13838
statusSource         0
screenName           0
retweetCount         0
isRetweet            0
retweeted            0
dtype: int64

In [28]:
# Drop Unnecessary Columns

columns_to_drop = [
    'replyToSN',
    'replyToSID',
    'replyToUID'
]

df.drop(columns=columns_to_drop, inplace=True)

print("Unnecessary Columns Removed ✅")

Unnecessary Columns Removed ✅


## Select Tweet Text Column

The `text` column contains the tweet content used for NLP preprocessing and sentiment analysis.

In [13]:
# Display Sample Tweets
df['text'].head()


0    RT @rssurjewala: Critical question: Was PayTM ...
1    RT @Hemant_80: Did you vote on #Demonetization...
2    RT @roshankar: Former FinSec, RBI Dy Governor,...
3    RT @ANI_news: Gurugram (Haryana): Post office ...
4    RT @satishacharya: Reddy Wedding! @mail_today ...
Name: text, dtype: object

## Initialize Stopwords and Lemmatizer

Stopwords are common words such as:
- is
- the
- and

which usually do not contribute significantly to sentiment analysis.

Lemmatization converts words into their root form.

In [14]:
# Initialize Stopwords
stop_words = set(stopwords.words('english'))

# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

print("Stopwords and Lemmatizer Ready ✅")

Stopwords and Lemmatizer Ready ✅


## Tweet Cleaning Function

This function performs:
- lowercase conversion
- URL removal
- mention removal
- hashtag removal
- punctuation removal
- tokenization
- stopword removal
- lemmatization

In [29]:
def clean_tweet(text):
    
    
    # Convert text to lowercase
    text = text.lower()

    # Remove RT (Retweet tag)
    text = re.sub(r'\brt\b', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    
    # Remove hashtags symbol only
    text = re.sub(r'#', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens
    cleaned_text = " ".join(tokens)
    
    return cleaned_text

## Apply Tweet Cleaning

The cleaning function is applied to all tweets to generate cleaned textual data for sentiment analysis.

In [30]:
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

nltk.download('punkt_tab')


df['cleaned_text'] = df['text'].astype(str).apply(clean_tweet)

print("Tweet Cleaning Completed ✅")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Tweet Cleaning Completed ✅


In [31]:
# Display Original vs Cleaned Tweets

df[['text', 'cleaned_text']].head()

,text,cleaned_text
0,RT @rssurjewala: Critical question: Was PayTM ...,critical question paytm informed demonetizatio...
1,RT @Hemant_80: Did you vote on #Demonetization...,vote demonetization modi survey app
2,"RT @roshankar: Former FinSec, RBI Dy Governor,...",former finsec rbi dy governor cbdt chair harva...
3,RT @ANI_news: Gurugram (Haryana): Post office ...,gurugram haryana post office employee provide ...
4,RT @satishacharya: Reddy Wedding! @mail_today ...,reddy wedding cartoon demonetization reddywedding


## Save Cleaned Dataset

The cleaned dataset is saved for further analysis and sentiment modeling in upcoming notebooks.

In [32]:
# Save Cleaned Dataset

df.to_csv("C:/Users/rahul/Desktop/demonetization-tweets.csv - Copy/demonetization-tweets.csv", index=False)

print("Cleaned Dataset Saved Successfully ✅")

Cleaned Dataset Saved Successfully ✅
